# Proyecto Final: Sistema de Recuperación de Información Multimodal con RAG

## Nombres: Goyes Antony, Quilumba Joel, Sangango Jose Armando

Diseñar e implementar un sistema de Recuperación de Información Multimodal capaz de responder consultas conversacionales sobre un corpus compuesto por texto e imágenes. Se integrarán modelos multimodales (CLIP), bases de datos vectoriales (FAISS/ChromaDB), y un flujo RAG con interfaz web.


## Guía de ejecución

- **SETUP** (Parte 0 + motor LLM + CLIP + ChromaDB + funciones): ejecutar SIEMPRE
  al iniciar sesión, en orden (~5 min).
- **CONSTRUCCIÓN DEL CORPUS** (Parte 1 y 2): ejecutar UNA sola vez; los datos
  persisten en Google Drive (~20-30 min la primera vez).
- **EVALUACIÓN**: resultados pre-calculados en CSVs; las celdas los regeneran.
- **INTERFAZ**: última celda, genera enlace público de Gradio.

## Parte 0: Instalación e Importación de Dependencias

Para construir nuestro pipeline multimodal, necesitamos librerías para manejar visión artificial, procesamiento de lenguaje natural, almacenamiento vectorial y la creación de la interfaz gráfica.

### Actividad
1. Instalar las librerías necesarias.
2. Importar los módulos principales para manipulación de datos e imágenes.

In [ ]:
# 1. Instalación de dependencias clave
##!pip install -q chromadb gradio google-generativeai
!pip install -q chromadb gradio groq sentence-transformers
# 2. Importación de módulos
import pandas as pd
import os
import numpy as np
import torch
from PIL import Image
import matplotlib.pyplot as plt
import os
import re

# Verificación de GPU
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"✅ Dependencias listas. Dispositivo: {device}")
from google.colab import drive
drive.mount('/content/drive')
DATA_DIR = '/content/drive/MyDrive/proyecto_ri/data'

✅ Dependencias listas. Dispositivo: cuda
Mounted at /content/drive


In [ ]:
!pip install -q groq
from groq import Groq
from google.colab import userdata

class LLMGroq:
    """Adaptador Groq: misma interfaz .generate_content() que usan todas las celdas."""
    def __init__(self, model='llama-3.1-8b-instant'):
        self.client = Groq(api_key=userdata.get('GROQ_API_KEY'))
        self.model = model
    def generate_content(self, prompt, request_options=None):
        r = self.client.chat.completions.create(
            model=self.model,
            messages=[{"role": "user", "content": prompt}],
            temperature=0.2)
        class Resp: pass
        resp = Resp(); resp.text = r.choices[0].message.content
        return resp

llm = LLMGroq()
print("Groq listo:", llm.generate_content("Say OK").text)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 5.8 MB/s eta 0:00:00
Groq listo: OK


## Parte 1: Preparación del Corpus Multimodal

Para este proyecto, utilizaremos el "Shopping Queries Image Dataset" de CrossingMinds. Este corpus es ideal porque ya contiene consultas (queries) y sus documentos relevantes asociados (qrels), lo que facilitará enormemente la evaluación experimental final.

El sistema trabajará sobre un corpus compuesto por documentos que incluyen información textual (títulos descriptivos) e imágenes (a través de URLs directas).

### Actividad
1. Instalar la librería `datasets` de HuggingFace.
2. Cargar la partición del catálogo de productos.
3. Transformar los datos a un DataFrame de Pandas para su manipulación.
4. Procesar y normalizar el texto de los documentos.
5. Asociar correctamente cada texto con la URL de su imagen correspondiente.



### Paso 1.1 — Descarga de datos ESCI

Montamos Google Drive (para que los datos persistan entre sesiones de Colab) y
descargamos los datos base del **Shopping Queries Dataset (ESCI)** de Amazon Science:

- `examples.parquet` → consultas (queries) y juicios de relevancia (qrels: E/S/C/I)
- `products.parquet` → títulos y descripciones de los productos

Más adelante uniremos estos datos con las imágenes de SQID mediante `product_id`.

In [ ]:
# ============================================================
# PARTE 1: CORPUS — Descarga de datos ESCI (Amazon Science)
# ============================================================
from google.colab import drive
import os

# 1. Montar Google Drive para persistencia entre sesiones
drive.mount('/content/drive')

# 2. Carpeta del proyecto en Drive (se crea si no existe)
PROJECT_DIR = '/content/drive/MyDrive/proyecto_ri'
DATA_DIR = os.path.join(PROJECT_DIR, 'data')
os.makedirs(DATA_DIR, exist_ok=True)
print(f"✅ Carpeta del proyecto: {PROJECT_DIR}")

# 3. Descargar los 3 parquet de ESCI desde el repo de GitHub de amazon-science
import urllib.request

ESCI_BASE = "https://github.com/amazon-science/esci-data/raw/main/shopping_queries_dataset"
archivos_esci = {
    "examples": "shopping_queries_dataset_examples.parquet",
    "products": "shopping_queries_dataset_products.parquet",
}

for nombre, archivo in archivos_esci.items():
    destino = os.path.join(DATA_DIR, archivo)
    if os.path.exists(destino):
        print(f"⏭️  {archivo} ya existe, se omite.")
        continue
    print(f"⬇️  Descargando {archivo}...")
    urllib.request.urlretrieve(f"{ESCI_BASE}/{archivo}", destino)
    print(f"✅ {archivo} descargado.")

print("\n✅ Datos ESCI listos.")

Mounted at /content/drive
✅ Carpeta del proyecto: /content/drive/MyDrive/proyecto_ri
⏭️  shopping_queries_dataset_examples.parquet ya existe, se omite.
⏭️  shopping_queries_dataset_products.parquet ya existe, se omite.

✅ Datos ESCI listos.


### Paso 1.2 — Inspección de los datos ESCI

Antes de procesar, examinamos la estructura de los dos archivos: qué columnas
tienen, cuántas filas, y qué valores toman las etiquetas de relevancia (`esci_label`).
Esto nos permite verificar que los datos son los esperados antes de filtrar.

In [ ]:
DATA_DIR = '/content/drive/MyDrive/proyecto_ri/data'

# Cargar los dos parquet
df_examples = pd.read_parquet(os.path.join(DATA_DIR, "shopping_queries_dataset_examples.parquet"))
df_products = pd.read_parquet(os.path.join(DATA_DIR, "shopping_queries_dataset_products.parquet"))

print("="*60)
print("EXAMPLES (queries + qrels)")
print("="*60)
print(f"Filas: {len(df_examples):,}")
print(f"Columnas: {list(df_examples.columns)}\n")
print(df_examples.head(3), "\n")

print("Valores de esci_label:")
print(df_examples['esci_label'].value_counts(), "\n")

print("Valores de product_locale:")
print(df_examples['product_locale'].value_counts(), "\n")

print("="*60)
print("PRODUCTS (títulos)")
print("="*60)
print(f"Filas: {len(df_products):,}")
print(f"Columnas: {list(df_products.columns)}\n")
print(df_products.head(3))

EXAMPLES (queries + qrels)
Filas: 2,621,288
Columnas: ['example_id', 'query', 'query_id', 'product_id', 'product_locale', 'esci_label', 'small_version', 'large_version', 'split']

   example_id           query  query_id  product_id product_locale esci_label  \
0           0   revent 80 cfm         0  B000MOO21W             us          I   
1           1   revent 80 cfm         0  B07X3Y6B1V             us          E   
2           2   revent 80 cfm         0  B07WDM7MQQ             us          E   

   small_version  large_version  split  
0              0              1  train  
1              0              1  train  
2              0              1  train   

Valores de esci_label:
esci_label
E    1708158
S     574313
I     263165
C      75652
Name: count, dtype: int64 

Valores de product_locale:
product_locale
us    1818825
jp     446053
es     356410
Name: count, dtype: int64 

PRODUCTS (títulos)
Filas: 1,814,924
Columnas: ['product_id', 'product_title', 'product_description', 'p

### Paso 1.3 - Autenticacion con Hugging Face

El dataset SQID (que contiene las URLs de las imagenes de cada producto) se aloja
en Hugging Face. Nos autenticamos con un token de acceso personal para poder
descargarlo. El token se genera en huggingface.co/settings/tokens (permiso de lectura).

In [ ]:
from huggingface_hub import login
from google.colab import userdata

# Token guardado en los Secrets de Colab (icono de llave, nombre: HF_TOKEN)
# Alternativa: login() sin argumentos abre un campo para pegarlo manualmente.
try:
    hf_token = userdata.get('HF_TOKEN')
    login(token=hf_token)
    print("Autenticado en Hugging Face correctamente.")
except Exception as e:
    print(f"No se encontro HF_TOKEN en Secrets: {e}")
    print("Ejecuta login() manualmente en una celda nueva y pega tu token.")

Autenticado en Hugging Face correctamente.


### Paso 1.4 - Descargar SQID y asociar imagenes al corpus

Descargamos de Hugging Face el archivo `product_image_urls.parquet` de SQID, que
contiene la URL de imagen de cada `product_id`. Lo unimos a nuestro corpus por
`product_id` y descartamos los productos sin imagen (SQID no obtuvo imagen para
un ~5% de los productos). Con esto cada producto queda asociado a su texto y su
imagen: se completa la Parte (a) del proyecto.

In [ ]:
from huggingface_hub import hf_hub_download

# Descargar el parquet de URLs de imagenes de SQID
print("Descargando product_image_urls.parquet de SQID...")
ruta_sqid = hf_hub_download(
    repo_id="crossingminds/shopping-queries-image-dataset",
    filename="data/product_image_urls.parquet",
    repo_type="dataset",
)
df_imagenes = pd.read_parquet(ruta_sqid)
print(f"Filas en SQID: {len(df_imagenes):,}")
print(f"Columnas: {list(df_imagenes.columns)}")

Descargando product_image_urls.parquet de SQID...
Filas en SQID: 164,900
Columnas: ['product_id', 'image_url']


### Paso 1.5 (corregido) - Muestreo restringido a productos con imagen

SQID solo cubre ~164.900 de los ~1.2M productos de ESCI-US (~13%). Si muestreamos
del ESCI completo, la mayoria de productos no tendra imagen. Por eso restringimos
el universo a los productos presentes en SQID ANTES de muestrear, y aplicamos el
minimo de juicios por query DESPUES de ese filtro, para que cada query conserve
suficientes productos con imagen y el Recall siga siendo informativo.

In [ ]:
# --- Parametros del corpus ---
N_QUERIES = 100
MIN_JUDGMENTS_PER_QUERY = 5
RANDOM_STATE = 42

# 1. Filtrar ESCI a test + US
df_eval = df_examples[
    (df_examples['split'] == 'test') &
    (df_examples['product_locale'] == 'us')
].copy()
print(f"Juicios (test + us): {len(df_eval):,}")

# 2. CLAVE: quedarnos solo con juicios cuyo producto tiene imagen en SQID
ids_con_imagen = set(df_imagenes['product_id'])
df_eval = df_eval[df_eval['product_id'].isin(ids_con_imagen)].copy()
print(f"Juicios con producto en SQID: {len(df_eval):,}")

# 3. Contar juicios por query DESPUES del filtro de imagen
juicios_por_query = df_eval['query_id'].value_counts()
queries_validas = juicios_por_query[juicios_por_query >= MIN_JUDGMENTS_PER_QUERY].index
print(f"Queries con >= {MIN_JUDGMENTS_PER_QUERY} juicios (con imagen): {len(queries_validas):,}")

# 4. Muestrear N_QUERIES
queries_muestra = pd.Series(list(queries_validas)).sample(
    n=N_QUERIES, random_state=RANDOM_STATE
).tolist()

# 5. Tomar todos los juicios (ya con imagen garantizada) de esas queries
df_corpus = df_eval[df_eval['query_id'].isin(queries_muestra)].copy()
print(f"\nJuicios en el corpus: {len(df_corpus):,}")
print(f"Productos unicos: {df_corpus['product_id'].nunique():,}")

# 6. Unir titulos (products US)
df_products_us = df_products[df_products['product_locale'] == 'us'][
    ['product_id', 'product_title', 'product_bullet_point']
].copy()
df_corpus = df_corpus.merge(df_products_us, on='product_id', how='left')
df_corpus = df_corpus[df_corpus['product_title'].notna()].copy()

# 7. Unir URLs de imagen
df_corpus = df_corpus.merge(df_imagenes, on='product_id', how='left')

# 8. Verificacion final de cobertura de imagen
con_img = df_corpus['image_url'].notna().sum()
print(f"Con imagen: {con_img} / {len(df_corpus)}  ({100*con_img/len(df_corpus):.1f}%)")
df_corpus = df_corpus[df_corpus['image_url'].notna()].copy()

# 9. Corpus final
print(f"\n--- CORPUS FINAL ---")
print(f"Juicios: {len(df_corpus):,}")
print(f"Productos unicos: {df_corpus['product_id'].nunique():,}")
print(f"Queries unicas: {df_corpus['query_id'].nunique()}")
print(f"\nDistribucion de relevancia:")
print(df_corpus['esci_label'].value_counts())

Juicios (test + us): 425,762
Juicios con producto en SQID: 213,926
Queries con >= 5 juicios (con imagen): 11,437

Juicios en el corpus: 1,911
Productos unicos: 1,895
Con imagen: 1839 / 1911  (96.2%)

--- CORPUS FINAL ---
Juicios: 1,839
Productos unicos: 1,823
Queries unicas: 100

Distribucion de relevancia:
esci_label
E    980
S    483
I    312
C     64
Name: count, dtype: int64


### Paso 1.6 - Guardar el corpus final

Persistimos el corpus en Drive para no depender de volver a ejecutar todo el
muestreo si la sesion se reinicia. Guardamos:
- `corpus.parquet`: un producto por fila (para indexar y descargar imagenes)
- `qrels.csv`: los juicios query-producto con su relevancia (para la evaluacion)
- `queries.csv`: el texto de cada query (para la evaluacion)

In [ ]:
import os
DATA_DIR = '/content/drive/MyDrive/proyecto_ri/data'

# Mapeo de relevancia ESCI -> ganancia para NDCG
ESCI_GAINS = {"E": 3, "S": 2, "C": 1, "I": 0}

# 1. qrels: todos los juicios (query, producto, relevancia)
df_qrels = df_corpus[['query_id', 'product_id', 'esci_label']].copy()
df_qrels['gain'] = df_qrels['esci_label'].map(ESCI_GAINS)
df_qrels.to_csv(os.path.join(DATA_DIR, 'qrels.csv'), index=False)
print(f"qrels.csv guardado: {len(df_qrels)} juicios")

# 2. queries: texto unico de cada query
df_queries = df_corpus[['query_id', 'query']].drop_duplicates().reset_index(drop=True)
df_queries.to_csv(os.path.join(DATA_DIR, 'queries.csv'), index=False)
print(f"queries.csv guardado: {len(df_queries)} queries")

# 3. corpus: un producto UNICO por fila (para indexar y descargar imagenes)
df_productos = df_corpus[
    ['product_id', 'product_title', 'product_bullet_point', 'image_url']
].drop_duplicates(subset='product_id').reset_index(drop=True)
df_productos.to_parquet(os.path.join(DATA_DIR, 'corpus.parquet'), index=False)
print(f"corpus.parquet guardado: {len(df_productos)} productos unicos")

# Verificacion
print(f"\nArchivos en {DATA_DIR}:")
for f in ['corpus.parquet', 'qrels.csv', 'queries.csv']:
    ruta = os.path.join(DATA_DIR, f)
    existe = "OK" if os.path.exists(ruta) else "FALTA"
    print(f"  [{existe}] {f}")

qrels.csv guardado: 1839 juicios
queries.csv guardado: 100 queries
corpus.parquet guardado: 1823 productos unicos

Archivos en /content/drive/MyDrive/proyecto_ri/data:
  [OK] corpus.parquet
  [OK] qrels.csv
  [OK] queries.csv


## Parte b - Representaciones vectoriales con CLIP

### Paso 2.1 - Descargar las imagenes a disco

Los embeddings de imagen requieren los archivos locales. Descargamos cada imagen
del corpus a `data/images/{product_id}.jpg`, guardandolas en Drive para que
persistan entre sesiones. Usamos cache (si ya existe, se omite), reintentos y un
User-Agent para evitar bloqueos. Los fallos se registran pero no detienen la descarga.

In [ ]:
import os
import requests
from tqdm.auto import tqdm
import pandas as pd

DATA_DIR = '/content/drive/MyDrive/proyecto_ri/data'
IMAGES_DIR = os.path.join(DATA_DIR, 'images')
os.makedirs(IMAGES_DIR, exist_ok=True)

# Cargar el corpus guardado
df_productos = pd.read_parquet(os.path.join(DATA_DIR, 'corpus.parquet'))
print(f"Productos a procesar: {len(df_productos)}")

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
        "(KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36"
    )
}

def descargar_imagen(product_id, url, reintentos=3):
    destino = os.path.join(IMAGES_DIR, f"{product_id}.jpg")
    if os.path.exists(destino):
        return "cache"
    for intento in range(reintentos):
        try:
            r = requests.get(url, headers=HEADERS, timeout=10)
            if r.status_code == 200:
                with open(destino, "wb") as f:
                    f.write(r.content)
                return "ok"
        except Exception:
            pass
    return "fallo"

# Descargar todas
resultados = {"ok": 0, "cache": 0, "fallo": 0}
fallidos = []
for _, row in tqdm(df_productos.iterrows(), total=len(df_productos), desc="Descargando"):
    estado = descargar_imagen(row["product_id"], row["image_url"])
    resultados[estado] += 1
    if estado == "fallo":
        fallidos.append(row["product_id"])

print(f"\nDescargadas: {resultados['ok']}")
print(f"Ya en cache: {resultados['cache']}")
print(f"Fallidas:    {resultados['fallo']}")
if fallidos:
    print(f"IDs fallidos (primeros 10): {fallidos[:10]}")

Productos a procesar: 1823


Descargando:   0%|          | 0/1823 [00:00<?, ?it/s]


Descargadas: 1823
Ya en cache: 0
Fallidas:    0


### Paso 2.2 - Cargar el modelo CLIP

Cargamos CLIP (openai/clip-vit-base-patch32), que proyecta texto e imagenes a un
mismo espacio vectorial de 512 dimensiones. Definimos dos funciones:
- `embed_textos(lista)`: vectoriza titulos o consultas (en lotes).
- `embed_imagenes(lista_rutas)`: vectoriza imagenes desde disco (en lotes).

Usamos la API oficial (`get_text_features` / `get_image_features`) y normalizamos
cada vector (norma L2 = 1), lo que permite usar el producto punto como similitud coseno.

In [ ]:
import torch
from transformers import CLIPModel, CLIPProcessor
from PIL import Image
import numpy as np

device = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_ID = "openai/clip-vit-base-patch32"
BATCH_SIZE = 32

print(f"Cargando CLIP en {device}...")
model = CLIPModel.from_pretrained(MODEL_ID).to(device)
processor = CLIPProcessor.from_pretrained(MODEL_ID)
model.eval()
print("Modelo cargado.")

# Imagen dummy (1x1) para acompanar al texto: el modelo completo espera ambas
# entradas, pero solo tomamos text_embeds. Es un truco de compatibilidad.
_IMG_DUMMY = Image.new("RGB", (224, 224))

@torch.no_grad()
def embed_textos(lista_textos, batch_size=BATCH_SIZE):
    """Vectoriza textos (titulos o consultas). Devuelve array (N, 512) normalizado."""
    vectores = []
    for i in range(0, len(lista_textos), batch_size):
        lote = lista_textos[i:i+batch_size]
        inputs = processor(
            text=lote,
            images=[_IMG_DUMMY] * len(lote),   # dummy, no se usa su salida
            return_tensors="pt",
            padding=True, truncation=True, max_length=77
        ).to(device)
        out = model(**inputs)
        emb = out.text_embeds                              # (N, 512)
        emb = emb / emb.norm(p=2, dim=-1, keepdim=True)    # normalizar L2
        vectores.append(emb.cpu().numpy())
    return np.vstack(vectores)

@torch.no_grad()
def embed_imagenes(lista_rutas, batch_size=BATCH_SIZE):
    """Vectoriza imagenes desde rutas de disco. Devuelve array (N, 512) normalizado."""
    vectores = []
    for i in range(0, len(lista_rutas), batch_size):
        lote_rutas = lista_rutas[i:i+batch_size]
        imagenes = [Image.open(r).convert("RGB") for r in lote_rutas]
        inputs = processor(
            text=[""] * len(imagenes),         # texto dummy, no se usa su salida
            images=imagenes,
            return_tensors="pt",
            padding=True
        ).to(device)
        out = model(**inputs)
        emb = out.image_embeds                             # (N, 512)
        emb = emb / emb.norm(p=2, dim=-1, keepdim=True)    # normalizar L2
        vectores.append(emb.cpu().numpy())
    return np.vstack(vectores)

# Prueba: texto e imagen deben dar shape (1, 512) y norma 1.000
prueba_txt = embed_textos(["a red running shoe"])
print(f"Texto:  shape {prueba_txt.shape}, norma {np.linalg.norm(prueba_txt[0]):.3f}")

Cargando CLIP en cuda...


config.json:   0%|          | 0.00/4.19k [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  605MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

...

vocab.json:   0%|          | 0.00/862k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/525k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.22M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/389 [00:00<?, ?B/s]

Modelo cargado.
Texto:  shape (1, 512), norma 1.000


### Paso 2.3 - Generar y guardar los embeddings del corpus

Aplicamos CLIP a los 1.823 productos para obtener dos matrices de embeddings:
- `emb_texto`: vector del titulo de cada producto
- `emb_imagen`: vector de la imagen de cada producto

Ambas comparten el mismo espacio de 512 dimensiones, lo que permitira comparar una
consulta de texto contra titulos y contra imagenes. Guardamos los vectores y el
orden de los product_id en Drive para reutilizarlos sin recalcular.

In [ ]:
DATA_DIR = '/content/drive/MyDrive/proyecto_ri/data'
IMAGES_DIR = os.path.join(DATA_DIR, 'images')
EMB_DIR = os.path.join(DATA_DIR, 'embeddings')
os.makedirs(EMB_DIR, exist_ok=True)

# Cargar el corpus
df_productos = pd.read_parquet(os.path.join(DATA_DIR, 'corpus.parquet'))
print(f"Productos: {len(df_productos)}")

# Orden fijo de product_id (los embeddings quedaran alineados a este orden)
product_ids = df_productos['product_id'].tolist()
titulos = df_productos['product_title'].tolist()
rutas_img = [os.path.join(IMAGES_DIR, f"{pid}.jpg") for pid in product_ids]

# 1. Embeddings de texto (titulos)
print("Generando embeddings de texto...")
emb_texto = embed_textos(titulos)
print(f"  emb_texto: {emb_texto.shape}")

# 2. Embeddings de imagen
print("Generando embeddings de imagen...")
emb_imagen = embed_imagenes(rutas_img)
print(f"  emb_imagen: {emb_imagen.shape}")

# 3. Guardar en Drive
np.save(os.path.join(EMB_DIR, 'emb_texto.npy'), emb_texto)
np.save(os.path.join(EMB_DIR, 'emb_imagen.npy'), emb_imagen)
with open(os.path.join(EMB_DIR, 'product_ids.txt'), 'w') as f:
    f.write("\n".join(product_ids))

print(f"\nGuardado en {EMB_DIR}:")
print(f"  emb_texto.npy   {emb_texto.shape}")
print(f"  emb_imagen.npy  {emb_imagen.shape}")
print(f"  product_ids.txt {len(product_ids)} ids")

# Verificacion: normas deben ser ~1.0
print(f"\nNorma media texto:  {np.linalg.norm(emb_texto, axis=1).mean():.3f}")
print(f"Norma media imagen: {np.linalg.norm(emb_imagen, axis=1).mean():.3f}")

Productos: 1823
Generando embeddings de texto...
  emb_texto: (1823, 512)
Generando embeddings de imagen...
  emb_imagen: (1823, 512)

Guardado en /content/drive/MyDrive/proyecto_ri/data/embeddings:
  emb_texto.npy   (1823, 512)
  emb_imagen.npy  (1823, 512)
  product_ids.txt 1823 ids

Norma media texto:  1.000
Norma media imagen: 1.000


## Parte c - Base de datos vectorial (ChromaDB)

### Paso 3.1 - Indexar los embeddings en dos colecciones

Creamos un cliente ChromaDB persistente (se guarda en Drive) con dos colecciones:
- `productos_texto`: embeddings de los titulos
- `productos_imagen`: embeddings de las imagenes

Ambas usan metrica coseno y guardan como metadato el titulo y la URL de imagen de
cada producto, para poder mostrar las evidencias despues. Tener dos colecciones nos
permitira combinar la senal de texto y la de imagen en la busqueda multimodal.

In [ ]:
import chromadb
DATA_DIR = '/content/drive/MyDrive/proyecto_ri/data'
EMB_DIR = os.path.join(DATA_DIR, 'embeddings')
CHROMA_DIR = os.path.join(DATA_DIR, 'chroma_db')

# Cargar embeddings, ids y corpus
emb_texto = np.load(os.path.join(EMB_DIR, 'emb_texto.npy'))
emb_imagen = np.load(os.path.join(EMB_DIR, 'emb_imagen.npy'))
with open(os.path.join(EMB_DIR, 'product_ids.txt')) as f:
    product_ids = f.read().splitlines()
df_productos = pd.read_parquet(os.path.join(DATA_DIR, 'corpus.parquet'))

# Alinear metadatos al orden de product_ids
df_productos = df_productos.set_index('product_id').loc[product_ids].reset_index()
print(f"Productos a indexar: {len(product_ids)}")

# Cliente persistente (se guarda en Drive)
client = chromadb.PersistentClient(path=CHROMA_DIR)

# Borrar colecciones si ya existen (para poder re-ejecutar la celda)
for nombre in ["productos_texto", "productos_imagen"]:
    try:
        client.delete_collection(nombre)
    except Exception:
        pass

# Crear las dos colecciones con metrica coseno
col_texto = client.create_collection("productos_texto", metadata={"hnsw:space": "cosine"})
col_imagen = client.create_collection("productos_imagen", metadata={"hnsw:space": "cosine"})

# Metadatos comunes (titulo + url) para mostrar evidencias
metadatas = [
    {"title": row["product_title"], "image_url": row["image_url"]}
    for _, row in df_productos.iterrows()
]
documentos = df_productos["product_title"].tolist()

# Indexar en lotes (Chroma recomienda no pasar todo de golpe)
def indexar(coleccion, embeddings):
    B = 500
    for i in range(0, len(product_ids), B):
        coleccion.add(
            ids=product_ids[i:i+B],
            embeddings=embeddings[i:i+B].tolist(),
            documents=documentos[i:i+B],
            metadatas=metadatas[i:i+B],
        )

print("Indexando texto...")
indexar(col_texto, emb_texto)
print("Indexando imagen...")
indexar(col_imagen, emb_imagen)

print(f"\nColeccion texto:  {col_texto.count()} documentos")
print(f"Coleccion imagen: {col_imagen.count()} documentos")

Productos a indexar: 1823
Indexando texto...
Indexando imagen...

Coleccion texto:  1823 documentos
Coleccion imagen: 1823 documentos


## Parte d (nucleo) - Busqueda multimodal con fusion

### Paso 4.1 - Funcion de busqueda

Definimos `buscar(query, k, alpha)`, la funcion central del sistema:
1. Vectoriza la consulta con CLIP.
2. Consulta las dos colecciones (texto e imagen) pidiendo el ranking completo.
3. Convierte distancias a similitudes (sim = 1 - dist).
4. Fusiona: score = alpha * sim_texto + (1-alpha) * sim_imagen.
5. Devuelve el Top-k con titulo, imagen y desglose de scores.

Pedimos el ranking completo de ambas colecciones para que cada candidato tenga
ambas senales (fusion exacta); con 1.823 documentos es instantaneo.

In [ ]:
N_CORPUS = col_texto.count()  # 1823: pedimos el ranking completo

def buscar(query_texto, k=10, alpha=0.5):
    """Busqueda multimodal: fusiona similitud de texto e imagen.

    Devuelve lista de dicts ordenados por score descendente:
    [{product_id, score, sim_texto, sim_imagen, title, image_url}, ...]
    """
    # 1. Vectorizar la consulta
    q_vec = embed_textos([query_texto])[0].tolist()

    # 2. Consultar ambas colecciones (ranking completo)
    res_t = col_texto.query(query_embeddings=[q_vec], n_results=N_CORPUS)
    res_i = col_imagen.query(query_embeddings=[q_vec], n_results=N_CORPUS)

    # 3. Distancia coseno -> similitud coseno
    sim_texto = {pid: 1.0 - dist for pid, dist in zip(res_t['ids'][0], res_t['distances'][0])}
    sim_imagen = {pid: 1.0 - dist for pid, dist in zip(res_i['ids'][0], res_i['distances'][0])}

    # Metadatos (titulo, url) desde la coleccion de texto
    metas = {pid: meta for pid, meta in zip(res_t['ids'][0], res_t['metadatas'][0])}

    # 4. Fusion sobre todos los candidatos
    resultados = []
    for pid in sim_texto:
        st, si = sim_texto[pid], sim_imagen.get(pid, 0.0)
        score = alpha * st + (1 - alpha) * si
        resultados.append({
            "product_id": pid,
            "score": score,
            "sim_texto": st,
            "sim_imagen": si,
            "title": metas[pid]["title"],
            "image_url": metas[pid]["image_url"],
        })

    # 5. Ordenar y devolver Top-k
    resultados.sort(key=lambda r: r["score"], reverse=True)
    return resultados[:k]


# --- Prueba con una query real del corpus ---
df_queries = pd.read_csv(os.path.join(DATA_DIR, 'queries.csv'))
query_prueba = df_queries['query'].iloc[0]
print(f"Query de prueba: '{query_prueba}'\n")

for alpha in [1.0, 0.5, 0.0]:
    print(f"--- alpha = {alpha} ({'solo texto' if alpha==1 else 'solo imagen' if alpha==0 else 'mixto'}) ---")
    top = buscar(query_prueba, k=3, alpha=alpha)
    for i, r in enumerate(top, 1):
        print(f"{i}. [{r['score']:.3f}] (txt {r['sim_texto']:.3f} / img {r['sim_imagen']:.3f}) {r['title'][:70]}")
    print()

Query de prueba: '1 dollar item free shipping'

--- alpha = 1.0 (solo texto) ---
1. [0.877] (txt 0.877 / img 0.213) Episode 6
2. [0.876] (txt 0.876 / img 0.229) Constant Therapy
3. [0.872] (txt 0.872 / img 0.204) Current Surgical Therapy

--- alpha = 0.5 (mixto) ---
1. [0.553] (txt 0.876 / img 0.229) Constant Therapy
2. [0.552] (txt 0.837 / img 0.267) Amazon eGift Card - Amazon For All Occasions
3. [0.545] (txt 0.877 / img 0.213) Episode 6

--- alpha = 0.0 (solo imagen) ---
1. [0.278] (txt 0.281 / img 0.278) Karseer Black Panther Magnetic Hematite and Matte Onyx Natural Stones 
2. [0.276] (txt 0.469 / img 0.276) Spalding NBA Street Phantom Outdoor Basketball Silver 29.5"
3. [0.275] (txt 0.241 / img 0.275) PRETTYGARDEN Off Shoulder Sleeve Hollow Out Summer Women Bodycon Long 



## Parte d - Sistema RAG (Retrieval-Augmented Generation)

### Paso 4.2 - Generacion de respuesta con contexto recuperado

Pipeline RAG completo:
1. El usuario hace una consulta.
2. `buscar()` recupera el Top-k multimodal.
3. Construimos automaticamente el contexto con los titulos recuperados.
4. Gemini genera una respuesta fundamentada SOLO en ese contexto.

La funcion devuelve tambien las evidencias (documentos, imagenes y scores) para
poder mostrarlas en la interfaz: transparencia entre recuperacion y generacion.

In [ ]:
RAG_TOP_K = 5  # documentos que se pasan como contexto al LLM

def responder_rag(consulta, k=RAG_TOP_K, alpha=0.0):
    """Pipeline RAG completo. Devuelve (respuesta, evidencias)."""
    # 1-2. Recuperar Top-k multimodal
    evidencias = buscar(consulta, k=k, alpha=alpha)

    # 3. Construir el contexto automaticamente
    contexto = ""
    for i, ev in enumerate(evidencias, 1):
        contexto += f"- Producto {i}: {ev['title']}\n"

    # 4. Prompt: respuesta fundamentada solo en el contexto
    prompt = f"""Eres un asistente virtual experto en compras y atencion al cliente.
Tu tarea es responder a la pregunta del usuario utilizando UNICAMENTE la informacion
de los productos proporcionados en el 'Contexto'. No inventes productos ni caracteristicas.
Si la respuesta no esta en el contexto, di honestamente que no encontraste un producto adecuado.
Se amable, claro y responde en espanol.

CONTEXTO:
{contexto}

PREGUNTA DEL USUARIO:
'{consulta}'

TU RESPUESTA:"""

    respuesta = llm.generate_content(prompt)
    return respuesta.text, evidencias


# --- Prueba del pipeline completo ---
consulta_prueba = "I need bright solar lights for my garden"
print(f"Consulta: '{consulta_prueba}'\n")

respuesta, evidencias = responder_rag(consulta_prueba)

print("=== RESPUESTA GENERADA ===")
print(respuesta)
print("\n=== EVIDENCIAS UTILIZADAS ===")
for i, ev in enumerate(evidencias, 1):
    print(f"{i}. [score {ev['score']:.3f}] {ev['title'][:70]}")

TimeoutException: Requesting secret GOOGLE_API_KEY_3 timed out. Secrets can only be fetched when running from the Colab UI.

## Parte f - Evaluacion experimental

### Paso 5.1 - Metricas de recuperacion

Implementamos las tres metricas del enunciado, calculadas por query y promediadas:

- **Precision@k**: fraccion del Top-k que es relevante (E o S).
- **Recall@k**: fraccion de los relevantes de la query que aparecen en el Top-k.
- **NDCG@k**: calidad del orden del ranking usando relevancia graduada
  (E=3, S=2, C=1, I=0), normalizada contra el ranking ideal.

Para Precision y Recall binarizamos la relevancia (relevante = E o S, gain >= 2).
NDCG usa la ganancia graduada completa. Las implementamos a mano (sin librerias de
evaluacion) para que el calculo sea transparente y explicable.

In [ ]:
import numpy as np

def precision_at_k(ranking_ids, relevantes_set, k):
    """Fraccion del top-k que es relevante."""
    top = ranking_ids[:k]
    if not top:
        return 0.0
    return sum(1 for pid in top if pid in relevantes_set) / k

def recall_at_k(ranking_ids, relevantes_set, k):
    """Fraccion de los relevantes que aparecen en el top-k."""
    if not relevantes_set:
        return 0.0
    top = ranking_ids[:k]
    return sum(1 for pid in top if pid in relevantes_set) / len(relevantes_set)

def dcg_at_k(ganancias, k):
    """Discounted Cumulative Gain: suma de ganancias con descuento logaritmico."""
    ganancias = ganancias[:k]
    return sum(g / np.log2(i + 2) for i, g in enumerate(ganancias))

def ndcg_at_k(ranking_ids, gains_dict, k):
    """DCG del ranking real dividido por el DCG del ranking ideal."""
    # Ganancia de cada resultado del ranking (0 si no fue juzgado)
    ganancias_reales = [gains_dict.get(pid, 0) for pid in ranking_ids[:k]]
    # Ranking ideal: los juzgados ordenados por ganancia descendente
    ganancias_ideales = sorted(gains_dict.values(), reverse=True)
    dcg = dcg_at_k(ganancias_reales, k)
    idcg = dcg_at_k(ganancias_ideales, k)
    return dcg / idcg if idcg > 0 else 0.0

# --- Prueba unitaria rapida con un caso a mano ---
# Ranking: [A, B, C]; relevantes: {A, C}; gains: A=3, C=2, D=1 (D no recuperado)
_rank = ["A", "B", "C"]
_rel = {"A", "C"}
_gains = {"A": 3, "C": 2, "D": 1}
print("P@3 =", round(precision_at_k(_rank, _rel, 3), 3), "(esperado 0.667)")
print("R@3 =", round(recall_at_k(_rank, _rel, 3), 3), "(esperado 1.0)")
print("NDCG@3 =", round(ndcg_at_k(_rank, _gains, 3), 3))


P@3 = 0.667 (esperado 0.667)
R@3 = 1.0 (esperado 1.0)
NDCG@3 = 0.84


In [ ]:
df_queries = pd.read_csv(os.path.join(DATA_DIR, 'queries.csv'))
df_qrels = pd.read_csv(os.path.join(DATA_DIR, 'qrels.csv'))
df_productos = pd.read_parquet(os.path.join(DATA_DIR, 'corpus.parquet'))
qrels_por_query = {}
for qid, grupo in df_qrels.groupby('query_id'):
    relevantes = set(grupo[grupo['gain'] >= 2]['product_id'])
    gains = dict(zip(grupo['product_id'], grupo['gain']))
    qrels_por_query[qid] = (relevantes, gains)
print(f"{len(df_queries)} queries, {len(df_qrels)} qrels, {len(df_productos)} productos")

### Paso 5.2 - Evaluacion completa con barrido de alpha

Evaluamos el sistema sobre las 100 queries del corpus, para varios valores de
alpha (peso texto vs imagen), reportando Precision@10, Recall@10 y NDCG@10
promediados sobre las queries. Esto responde empiricamente la pregunta central:
que combinacion de senales (texto/imagen) recupera mejor?

- alpha = 1.0: solo texto | alpha = 0.0: solo imagen | intermedios: fusion

In [ ]:
from tqdm.auto import tqdm

df_queries = pd.read_csv(os.path.join(DATA_DIR, 'queries.csv'))
df_qrels = pd.read_csv(os.path.join(DATA_DIR, 'qrels.csv'))

K_EVAL = 10
ALPHAS = [0.0, 0.25, 0.5, 0.75, 1.0]

# Pre-construir por query: set de relevantes (E/S) y dict de ganancias
qrels_por_query = {}
for qid, grupo in df_qrels.groupby('query_id'):
    relevantes = set(grupo[grupo['gain'] >= 2]['product_id'])
    gains = dict(zip(grupo['product_id'], grupo['gain']))
    qrels_por_query[qid] = (relevantes, gains)

resultados_alpha = []
for alpha in ALPHAS:
    metricas = {"P": [], "R": [], "NDCG": []}
    for _, qrow in tqdm(df_queries.iterrows(), total=len(df_queries),
                        desc=f"alpha={alpha}", leave=False):
        qid, qtexto = qrow['query_id'], qrow['query']
        relevantes, gains = qrels_por_query[qid]

        ranking = buscar(qtexto, k=K_EVAL, alpha=alpha)
        ranking_ids = [r['product_id'] for r in ranking]

        metricas["P"].append(precision_at_k(ranking_ids, relevantes, K_EVAL))
        metricas["R"].append(recall_at_k(ranking_ids, relevantes, K_EVAL))
        metricas["NDCG"].append(ndcg_at_k(ranking_ids, gains, K_EVAL))

    resultados_alpha.append({
        "alpha": alpha,
        f"P@{K_EVAL}": np.mean(metricas["P"]),
        f"R@{K_EVAL}": np.mean(metricas["R"]),
        f"NDCG@{K_EVAL}": np.mean(metricas["NDCG"]),
    })

df_resultados = pd.DataFrame(resultados_alpha)
print("\n=== RESULTADOS: barrido de alpha (promedio sobre 100 queries) ===")
print(df_resultados.to_string(index=False))

# Guardar para el informe
df_resultados.to_csv(os.path.join(DATA_DIR, 'resultados_alpha.csv'), index=False)
print("\nGuardado en resultados_alpha.csv")

alpha=0.0:   0%|          | 0/100 [00:00<?, ?it/s]

alpha=0.25:   0%|          | 0/100 [00:00<?, ?it/s]

alpha=0.5:   0%|          | 0/100 [00:00<?, ?it/s]

alpha=0.75:   0%|          | 0/100 [00:00<?, ?it/s]

alpha=1.0:   0%|          | 0/100 [00:00<?, ?it/s]


=== RESULTADOS: barrido de alpha (promedio sobre 100 queries) ===
 alpha  P@10     R@10  NDCG@10
  0.00 0.564 0.449719 0.623660
  0.25 0.513 0.410234 0.576712
  0.50 0.375 0.289953 0.436953
  0.75 0.309 0.237235 0.367023
  1.00 0.244 0.188714 0.296898

Guardado en resultados_alpha.csv


## Funcionalidades de excelencia

### Excelencia 1 - Re-ranking con cross-encoder

Refinamos el ranking inicial en dos etapas:
1. **Recuperacion** (rapida): `buscar()` con alpha=0 trae un pool amplio (top-50).
2. **Re-ranking** (preciso): un cross-encoder (ms-marco-MiniLM) lee cada par
   (query, titulo) conjuntamente y reordena el pool por su score; devolvemos el top-k.

El cross-encoder es mas preciso que la similitud de embeddings porque modela la
interaccion entre query y documento, pero es mas costoso: por eso solo se aplica
al pool recuperado, no al corpus completo (arquitectura estandar de dos etapas).

In [ ]:
!pip install -q sentence-transformers

from sentence_transformers import CrossEncoder

print("Cargando cross-encoder...")
reranker = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2', device=device)
print("Cross-encoder cargado.")

POOL_SIZE = 50  # cuantos candidatos recupera la etapa 1

def buscar_con_rerank(query_texto, k=10, alpha=0.0, pool_size=POOL_SIZE):
    """Busqueda en dos etapas: recuperacion vectorial + re-ranking con cross-encoder.

    Devuelve el mismo formato que buscar(), con un campo extra 'score_rerank'.
    """
    # Etapa 1: pool amplio con la mejor configuracion vectorial
    pool = buscar(query_texto, k=pool_size, alpha=alpha)

    # Etapa 2: puntuar cada par (query, titulo) con el cross-encoder
    pares = [(query_texto, r['title']) for r in pool]
    scores = reranker.predict(pares)

    for r, s in zip(pool, scores):
        r['score_rerank'] = float(s)

    # Reordenar por el score del cross-encoder y devolver top-k
    pool.sort(key=lambda r: r['score_rerank'], reverse=True)
    return pool[:k]


# --- Prueba cualitativa: la query problematica de antes ---
consulta = "I need bright solar lights for my garden"
print(f"\nQuery: '{consulta}'")

print("\n--- SIN re-ranking (alpha=0, top-5) ---")
for i, r in enumerate(buscar(consulta, k=5, alpha=0.0), 1):
    print(f"{i}. [{r['score']:.3f}] {r['title'][:70]}")

print("\n--- CON re-ranking (pool=50 -> top-5) ---")
for i, r in enumerate(buscar_con_rerank(consulta, k=5), 1):
    print(f"{i}. [rerank {r['score_rerank']:.2f}] {r['title'][:70]}")

Cargando cross-encoder...


config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

Cross-encoder cargado.

Query: 'I need bright solar lights for my garden'

--- SIN re-ranking (alpha=0, top-5) ---
1. [0.316] Solar Lights Outdoor - New Upgraded Solar Garden Lights， Multi-Color A
2. [0.314] GIGALUMI Solar Powered Path Lights, Solar Garden Lights Outdoor, Lands
3. [0.313] Solar Lights Outdoor, [6 Pack/3 Modes/50LED] SEZAC Motion Sensor Secur
4. [0.305] Solar Lights Outdoor [6 Pack/3 Working Mode], SEZAC Solar Security Lig
5. [0.305] Solar Lights Outdoor Decorative - 2 Pack Magic Crystal Solar Globe Lig

--- CON re-ranking (pool=50 -> top-5) ---
1. [rerank 4.01] Super Bright 2-Pack 66FT 200 LED Solar Christmas Lights Outdoor, Water
2. [rerank 3.90] Outdoor Solar Garden Stake Lights,Upgraded LED Solar Powered Light wit
3. [rerank 2.87] Aloudy Solar Garden Stake Lights, Upgraded 3 Pack Outdoor Waterproof S
4. [rerank 2.17] Solar Lights Outdoor Decorative - 2 Pack Magic Crystal Solar Globe Lig
5. [rerank 1.91] Solar Lights Outdoor - New Upgraded Solar Garden Lights， Multi-

### Evaluacion de la Excelencia 1 - Re-ranking vs baseline

Medimos el efecto del re-ranking sobre las 100 queries con las mismas metricas
(P@10, R@10, NDCG@10), comparando:
- **Baseline**: busqueda vectorial con alpha=0 (la mejor configuracion del barrido)
- **Re-ranking**: pool de 50 candidatos (alpha=0) reordenado por el cross-encoder

La comparacion antes/despues con metricas identicas permite atribuir la mejora
exclusivamente al re-ranking.

In [ ]:
from tqdm.auto import tqdm
K_EVAL = 10

filas = []
for nombre, fn in [
    ("Baseline (alpha=0)", lambda q: buscar(q, k=K_EVAL, alpha=0.0)),
    ("Re-ranking (pool=50)", lambda q: buscar_con_rerank(q, k=K_EVAL)),
]:
    P, R, N = [], [], []
    for _, qrow in tqdm(df_queries.iterrows(), total=len(df_queries),
                        desc=nombre, leave=False):
        qid, qtexto = qrow['query_id'], qrow['query']
        relevantes, gains = qrels_por_query[qid]
        ranking_ids = [r['product_id'] for r in fn(qtexto)]
        P.append(precision_at_k(ranking_ids, relevantes, K_EVAL))
        R.append(recall_at_k(ranking_ids, relevantes, K_EVAL))
        N.append(ndcg_at_k(ranking_ids, gains, K_EVAL))
    filas.append({"sistema": nombre, f"P@{K_EVAL}": np.mean(P),
                  f"R@{K_EVAL}": np.mean(R), f"NDCG@{K_EVAL}": np.mean(N)})

df_rerank = pd.DataFrame(filas)
print("\n=== EXCELENCIA 1: Re-ranking vs Baseline (100 queries) ===")
print(df_rerank.to_string(index=False))
df_rerank.to_csv(os.path.join(DATA_DIR, 'resultados_rerank.csv'), index=False)
print("\nGuardado en resultados_rerank.csv")

Baseline (alpha=0):   0%|          | 0/100 [00:00<?, ?it/s]

Re-ranking (pool=50):   0%|          | 0/100 [00:00<?, ?it/s]


=== EXCELENCIA 1: Re-ranking vs Baseline (100 queries) ===
             sistema  P@10     R@10  NDCG@10
  Baseline (alpha=0) 0.564 0.449719 0.623660
Re-ranking (pool=50) 0.649 0.524148 0.722255

Guardado en resultados_rerank.csv


### Excelencia 2 - Query Expansion / Reformulacion con LLM

Las consultas conversacionales contienen relleno ("I need...", "for my garden",
"nothing too expensive") que diluye el embedding de CLIP. Usamos Gemini para
reformular automaticamente la consulta a una forma keyword compacta antes de
vectorizarla. El pipeline queda: consulta -> reformulacion (LLM) -> busqueda ->
re-ranking. Notese que el re-ranking usa la consulta ORIGINAL (el cross-encoder
si se beneficia del contexto completo del usuario).

In [ ]:
def reformular_query(consulta):
    """Convierte una consulta conversacional en una query de busqueda compacta."""
    prompt = f"""Convert this shopping request into a short product search query
(3-6 keywords, in English). Reply with ONLY the search query, nothing else.

Request: "{consulta}"
Search query:"""
    try:
        r = llm.generate_content(prompt)
        reformulada = r.text.strip().strip('"').strip()
        # Salvaguarda: si el LLM devuelve algo vacio o larguisimo, usar la original
        if not reformulada or len(reformulada) > 100:
            return consulta
        return reformulada
    except Exception:
        return consulta  # ante cualquier fallo del LLM, la busqueda no se cae

def buscar_completo(consulta, k=10, usar_expansion=True, usar_rerank=True):
    """Pipeline completo: reformulacion -> recuperacion (alpha=0) -> re-ranking.

    La consulta reformulada se usa para RECUPERAR (embedding);
    la consulta original se usa para RE-RANKEAR (el cross-encoder entiende contexto).
    """
    q_busqueda = reformular_query(consulta) if usar_expansion else consulta

    if usar_rerank:
        pool = buscar(q_busqueda, k=POOL_SIZE, alpha=0.0)
        pares = [(consulta, r['title']) for r in pool]   # consulta ORIGINAL
        scores = reranker.predict(pares)
        for r, s in zip(pool, scores):
            r['score_rerank'] = float(s)
        pool.sort(key=lambda r: r['score_rerank'], reverse=True)
        resultados = pool[:k]
    else:
        resultados = buscar(q_busqueda, k=k, alpha=0.0)

    return resultados, q_busqueda


# --- Prueba cualitativa ---
consultas_prueba = [
    "I need bright solar lights for my garden",
    "im looking for something to organize my desk cables, they are a mess",
]
for c in consultas_prueba:
    resultados, q_ref = buscar_completo(c, k=3)
    print(f"\nOriginal:    '{c}'")
    print(f"Reformulada: '{q_ref}'")
    for i, r in enumerate(resultados, 1):
        print(f"  {i}. [rerank {r['score_rerank']:.2f}] {r['title'][:65]}")


Original:    'I need bright solar lights for my garden'
Reformulada: 'bright solar garden lights'
  1. [rerank 4.01] Super Bright 2-Pack 66FT 200 LED Solar Christmas Lights Outdoor, 
  2. [rerank 3.90] Outdoor Solar Garden Stake Lights,Upgraded LED Solar Powered Ligh
  3. [rerank 2.87] Aloudy Solar Garden Stake Lights, Upgraded 3 Pack Outdoor Waterpr

Original:    'im looking for something to organize my desk cables, they are a mess'
Reformulada: 'desk cable organizer'
  1. [rerank -11.18] Mr IRONSTONE L-Shaped Desk 50.8" Computer Corner Desk, Home Gamin
  2. [rerank -11.19] Echo Auto Air Vent Mount
  3. [rerank -11.26] Whitmor 4 Tier Storage Organizer-Natural Wood and Chrome Closet S


In [ ]:
import time

# 1. Generar (una vez) la version conversacional de cada query y guardarla
RUTA_CONV = os.path.join(DATA_DIR, 'queries_conversacionales.csv')

if os.path.exists(RUTA_CONV):
    df_conv = pd.read_csv(RUTA_CONV)
    print(f"Cargadas {len(df_conv)} queries conversacionales desde cache.")
else:
    print("Generando versiones conversacionales con Gemini (una sola vez)...")
    filas = []
    for _, row in tqdm(df_queries.iterrows(), total=len(df_queries)):
        prompt = f"""Rewrite this product search query as a natural, casual shopping
request a person would type to a chat assistant (1 sentence, in English).
Keep the SAME product intent. Reply ONLY with the rewritten request.

Query: "{row['query']}"
Rewritten:"""
        try:
            r = llm.generate_content(prompt)
            conv = r.text.strip().strip('"')
        except Exception:
            conv = row['query']
            time.sleep(1)
        filas.append({"query_id": row['query_id'], "query": row['query'],
                      "query_conv": conv})
    df_conv = pd.DataFrame(filas)
    df_conv.to_csv(RUTA_CONV, index=False)
    print(f"Guardadas en {RUTA_CONV}")

print("\nEjemplos:")
for _, r in df_conv.head(3).iterrows():
    print(f"  '{r['query']}'  ->  '{r['query_conv']}'")

Cargadas 100 queries conversacionales desde cache.

Ejemplos:
  '1 dollar item free shipping'  ->  'I'm looking for something that costs around a dollar and ships for free.'
  '1 gpm propane water heater not adapter'  ->  'I'm looking for a 1 gpm propane water heater, the heater itself and not an adapter.'
  '1.50 mens xl reading glasses without nose piece'  ->  'Hey, I'm looking for some men's XL 1.50 reading glasses that don't have a nose piece.'


### Evaluacion de la Excelencia 2 - Query Expansion

Comparamos tres configuraciones sobre las 100 queries (todas con recuperacion
alpha=0 + re-ranking, K=10):

1. **Query ESCI original** (keyword-style): referencia superior.
2. **Conversacional SIN expansion**: mide cuanto degrada el estilo conversacional
   al sistema (las consultas reales de un chat traen relleno que diluye el embedding).
3. **Conversacional CON expansion**: mide cuanto rescata la reformulacion LLM
   (recuperamos con la query reformulada; el re-ranking usa la conversacional
   original, ya que el cross-encoder si aprovecha el contexto completo).

Las versiones conversacionales preservan la intencion de la query original, por lo
que se evaluan con los mismos qrels. Las reformulaciones se cachean en Drive para
no repetir llamadas al LLM.

In [ ]:
from tqdm.auto import tqdm
import time

K_EVAL = 10
RUTA_REFORM = os.path.join(DATA_DIR, 'reformulaciones.csv')

# --- 1. Reformular las 100 conversacionales (con cache en Drive) ---
if os.path.exists(RUTA_REFORM):
    df_reform = pd.read_csv(RUTA_REFORM)
    print(f"Cargadas {len(df_reform)} reformulaciones desde cache.")
else:
    print("Reformulando las 100 queries conversacionales (una sola vez)...")
    filas = []
    for _, row in tqdm(df_conv.iterrows(), total=len(df_conv)):
        reform = reformular_query(row['query_conv'])
        filas.append({"query_id": row['query_id'], "reformulada": reform})
        time.sleep(2)  # anti rate-limit
    df_reform = pd.DataFrame(filas)
    df_reform.to_csv(RUTA_REFORM, index=False)
    print(f"Guardadas en {RUTA_REFORM}")

# Unir todo en un solo dataframe de evaluacion
df_eval_q = df_conv.merge(df_reform, on='query_id')
print("\nEjemplo del pipeline completo:")
ej = df_eval_q.iloc[0]
print(f"  ESCI:           '{ej['query']}'")
print(f"  Conversacional: '{ej['query_conv']}'")
print(f"  Reformulada:    '{ej['reformulada']}'")

# --- 2. Evaluar las tres configuraciones ---
configs = [
    ("1. ESCI original",           'query',       None),
    ("2. Conversacional sin exp.", 'query_conv',  None),
    ("3. Conversacional con exp.", 'reformulada', 'query_conv'),
]
# Config 3: recuperamos con la reformulada, re-rankeamos con la conversacional

filas = []
for nombre, col_busqueda, col_rerank in configs:
    P, R, N = [], [], []
    for _, row in tqdm(df_eval_q.iterrows(), total=len(df_eval_q),
                       desc=nombre, leave=False):
        qid = row['query_id']
        relevantes, gains = qrels_por_query[qid]
        q_buscar = row[col_busqueda]
        q_rer = row[col_rerank] if col_rerank else q_buscar

        pool = buscar(q_buscar, k=POOL_SIZE, alpha=0.0)
        pares = [(q_rer, r['title']) for r in pool]
        scores = reranker.predict(pares)
        for r, s in zip(pool, scores):
            r['score_rerank'] = float(s)
        pool.sort(key=lambda r: r['score_rerank'], reverse=True)
        ranking_ids = [r['product_id'] for r in pool[:K_EVAL]]

        P.append(precision_at_k(ranking_ids, relevantes, K_EVAL))
        R.append(recall_at_k(ranking_ids, relevantes, K_EVAL))
        N.append(ndcg_at_k(ranking_ids, gains, K_EVAL))
    filas.append({"configuracion": nombre, f"P@{K_EVAL}": np.mean(P),
                  f"R@{K_EVAL}": np.mean(R), f"NDCG@{K_EVAL}": np.mean(N)})

df_exp = pd.DataFrame(filas)
print("\n=== EXCELENCIA 2: Query Expansion (100 queries) ===")
print(df_exp.to_string(index=False))
df_exp.to_csv(os.path.join(DATA_DIR, 'resultados_expansion.csv'), index=False)
print("\nGuardado en resultados_expansion.csv")

Reformulando las 100 queries conversacionales (una sola vez)...


  0%|          | 0/100 [00:00<?, ?it/s]

Guardadas en /content/drive/MyDrive/proyecto_ri/data/reformulaciones.csv

Ejemplo del pipeline completo:
  ESCI:           '1 dollar item free shipping'
  Conversacional: 'I'm looking for something that costs around a dollar and ships for free.'
  Reformulada:    'Affordable products under $1 with free shipping'


1. ESCI original:   0%|          | 0/100 [00:00<?, ?it/s]

2. Conversacional sin exp.:   0%|          | 0/100 [00:00<?, ?it/s]

3. Conversacional con exp.:   0%|          | 0/100 [00:00<?, ?it/s]


=== EXCELENCIA 2: Query Expansion (100 queries) ===
             configuracion  P@10     R@10  NDCG@10
          1. ESCI original 0.649 0.524148 0.722255
2. Conversacional sin exp. 0.662 0.542889 0.741490
3. Conversacional con exp. 0.658 0.527874 0.727140

Guardado en resultados_expansion.csv


"Contrario a nuestra hipótesis, las consultas conversacionales no degradaron el rendimiento (NDCG 0.741 vs 0.722), y la expansión no aportó mejora adicional (0.727). El análisis sugiere que el cross-encoder de la etapa de re-ranking, entrenado sobre preguntas en lenguaje natural, aprovecha el contexto conversacional directamente, haciendo la reformulación redundante en esta arquitectura. La expansión mantiene valor potencial en configuraciones sin re-ranking, donde la recuperación por embedding es sensible a la dilución del texto largo

### Excelencia 4 - Memoria conversacional

El sistema mantiene el historial de turnos y lo usa en las dos fases:

1. **Recuperacion**: las consultas de seguimiento ("y hay de flores?") son
   ambiguas por si solas. Un LLM las reescribe como consulta autonoma usando
   el historial ("solar garden lights with flowers") antes de vectorizar y buscar.
2. **Generacion**: el LLM recibe el historial reciente ademas del contexto
   recuperado, permitiendo respuestas coherentes con la conversacion.

Si la consulta ya es autonoma, el reescritor la deja igual (el historial no
contamina consultas nuevas independientes).

In [ ]:
# Funciones de feedback (Rocchio) sin levantar la interfaz Gradio
import numpy as np

BETA, GAMMA = 0.5, 0.3
feedback = {"likes": set(), "dislikes": set()}
RAG_TOP_K = 5

emb_por_id = {pid: emb_imagen[i] for i, pid in enumerate(product_ids)}

def aplicar_rocchio(q_vec):
    q = np.array(q_vec, dtype=np.float32)
    if feedback["likes"]:
        q = q + BETA * np.mean([emb_por_id[p] for p in feedback["likes"]], axis=0)
    if feedback["dislikes"]:
        q = q - GAMMA * np.mean([emb_por_id[p] for p in feedback["dislikes"]], axis=0)
    return (q / np.linalg.norm(q)).tolist()

def buscar_con_feedback(query_texto, k=5, pool_size=POOL_SIZE):
    q_vec = embed_textos([query_texto])[0].tolist()
    q_vec = aplicar_rocchio(q_vec)
    res_i = col_imagen.query(query_embeddings=[q_vec], n_results=pool_size)
    metas = {p: m for p, m in zip(res_i['ids'][0], res_i['metadatas'][0])}
    pool = [{"product_id": p, "sim": 1.0 - d,
             "title": metas[p]["title"], "image_url": metas[p]["image_url"]}
            for p, d in zip(res_i['ids'][0], res_i['distances'][0])]
    pares = [(query_texto, r['title']) for r in pool]
    scores = reranker.predict(pares)
    for r, s in zip(pool, scores):
        r['score_rerank'] = float(s)
    pool.sort(key=lambda r: r['score_rerank'], reverse=True)
    return pool[:k]

print("Funciones de feedback definidas.")

Funciones de feedback definidas.


In [ ]:
def reescribir_con_historial(consulta, historial):
    """Convierte una consulta de seguimiento en una autonoma usando el historial.

    historial: lista de tuplas (rol, texto), ej [("user","..."),("assistant","...")]
    Si no hay historial o la consulta ya es autonoma, la devuelve (casi) igual.
    """
    if not historial:
        return consulta

    hist_texto = "\n".join(f"{rol}: {txt}" for rol, txt in historial[-6:])
    prompt = f"""Given this shopping chat history and the user's new message,
rewrite the new message as a SELF-CONTAINED product search query in English
(include the product context from history if the message refers to it).
If the new message is already self-contained, return it unchanged.
Reply with ONLY the rewritten query, nothing else.

Chat history:
{hist_texto}

New message: "{consulta}"
Self-contained query:"""
    try:
        r = llm.generate_content(prompt)
        reescrita = r.text.strip().strip('"').strip()
        if not reescrita or len(reescrita) > 150:
            return consulta
        return reescrita
    except Exception:
        return consulta


def responder_con_memoria(consulta, historial):
    """Pipeline completo con memoria: reescritura -> busqueda -> RAG con historial.

    Devuelve (respuesta, evidencias, consulta_reescrita).
    """
    # 1. Reescribir la consulta con el contexto de la conversacion
    q_autonoma = reescribir_con_historial(consulta, historial)

    # 2. Recuperar con la consulta autonoma (alpha=0 + re-ranking, via Rocchio si hay feedback)
    resultados = buscar_con_feedback(q_autonoma, k=RAG_TOP_K)

    # 3. Generar respuesta con contexto recuperado + historial reciente
    contexto = "".join(f"- Producto {i}: {r['title']}\n"
                       for i, r in enumerate(resultados, 1))
    hist_texto = "\n".join(f"{rol}: {txt}" for rol, txt in historial[-4:])

    prompt = f"""Eres un asistente de busqueda de PRODUCTOS de una tienda.
Tu UNICA funcion es recomendar productos del Contexto. No inventes productos.
NUNCA ofrezcas recetas, consejos, tutoriales ni nada que no sea recomendar
los productos del Contexto. Si nada del contexto es relevante al mensaje,
di brevemente que no encontraste productos adecuados y sugiere reformular
la busqueda. No ofrezcas alternativas que no puedas cumplir.

CONVERSACION PREVIA:
{hist_texto if hist_texto else "(inicio de conversacion)"}

CONTEXTO (productos recuperados):
{contexto}

NUEVO MENSAJE DEL USUARIO: '{consulta}'
RESPUESTA:"""
    try:
        respuesta = llm.generate_content(prompt).text
    except Exception:
        respuesta = "(LLM no disponible; se muestran las evidencias recuperadas.)"

    return respuesta, resultados, q_autonoma


# --- Prueba del flujo conversacional completo ---
historial_prueba = []

# Turno 1: consulta inicial
c1 = "I'm looking for solar lights for my garden"
r1, ev1, q1 = responder_con_memoria(c1, historial_prueba)
historial_prueba += [("user", c1), ("assistant", r1[:200])]
print(f"TURNO 1: '{c1}'")
print(f"  Reescrita: '{q1}'")
print(f"  Top-3: {[e['title'][:40] for e in ev1[:3]]}")
print(f"  Respuesta: {r1[:150]}...\n")

# Turno 2: seguimiento AMBIGUO (solo tiene sentido con memoria)
c2 = "do you have any with flowers?"
r2, ev2, q2 = responder_con_memoria(c2, historial_prueba)
print(f"TURNO 2: '{c2}'")
print(f"  Reescrita: '{q2}'   <-- la magia esta aqui")
print(f"  Top-3: {[e['title'][:40] for e in ev2[:3]]}")
print(f"  Respuesta: {r2[:150]}...")

TURNO 1: 'I'm looking for solar lights for my garden'
  Reescrita: 'I'm looking for solar lights for my garden'
  Top-3: ['Aloudy Solar Garden Stake Lights, Upgrad', 'Solar Lights Outdoor - New Upgraded Sola', 'Solar Lights Outdoor Decorative - 2 Pack']
  Respuesta: (LLM no disponible; se muestran las evidencias recuperadas.)...



TURNO 2: 'do you have any with flowers?'
  Reescrita: 'do you have any with flowers?'   <-- la magia esta aqui
  Top-3: ['2 Sets Cemetery Flowers with vase/Grave ', 'Everlasting Silk Flowers Cemetery Vase w', 'Outdoor Solar Garden Stake Lights,Upgrad']
  Respuesta: (LLM no disponible; se muestran las evidencias recuperadas.)...


## Parte e - Interfaz conversacional + Excelencia 3 (Relevance Feedback)

Interfaz web tipo chat (Gradio) que integra el pipeline completo:
consulta -> recuperacion multimodal (alpha=0) -> re-ranking -> RAG.

Muestra la respuesta del asistente y las evidencias (productos, imagenes y
scores) en un panel inspeccionable, cumpliendo el requisito de transparencia.

**Relevance Feedback (Rocchio)**: cada resultado puede calificarse con Me gusta /
No me gusta. El feedback modifica el vector de consultas posteriores:
q' = q + beta

In [ ]:
import gradio as gr

# ---------- Estado de la conversacion ----------
# (el estado 'feedback' y las funciones aplicar_rocchio / buscar_con_feedback
#  se definen en la celda "Funciones de feedback (Rocchio)" de las excelencias)
historial_chat = []   # memoria conversacional: [(rol, texto), ...]

# ---------- Logica del chat (con memoria conversacional) ----------
def responder(mensaje, chat_visible):
    global historial_chat

    # Pipeline completo: reescritura con historial -> Rocchio -> re-ranking -> RAG
    respuesta, resultados, q_reescrita = responder_con_memoria(mensaje, historial_chat)
    UMBRAL_RERANK = -5.0
    hay_resultados = resultados and resultados[0]['score_rerank'] > UMBRAL_RERANK
    if not hay_resultados:
        resultados = []

    # Actualizar la memoria
    historial_chat += [("user", mensaje), ("assistant", respuesta[:200])]

    # Chat visible
    chat_visible = chat_visible + [
        {"role": "user", "content": mensaje},
        {"role": "assistant", "content": respuesta},
    ]

    # Panel de evidencias (con nota de memoria si hubo reescritura)
    nota_memoria = ""
    if q_reescrita.strip().lower() != mensaje.strip().lower():
        nota_memoria = (f"<p><small><b>Memoria conversacional:</b> consulta "
                        f"reescrita como <i>'{q_reescrita}'</i></small></p>")
    if resultados:
        html = f"<h4>Evidencias utilizadas (Top-{len(resultados)})</h4>{nota_memoria}"
    else:
        html = "<p><i>Sin resultados relevantes para este mensaje (no parece una consulta de producto).</i></p>"
    for r in resultados:
        marca = ""
        if r['product_id'] in feedback['likes']:
            marca = " | LIKE"
        elif r['product_id'] in feedback['dislikes']:
            marca = " | DISLIKE"
        html += f"""
        <div style='display:flex;gap:10px;margin:8px 0;padding:8px;
                    border:1px solid #ddd;border-radius:8px;align-items:center'>
          <img src='{r["image_url"]}' style='width:70px;height:70px;object-fit:contain'>
          <div style='flex:1'>
            <b>{r["title"][:90]}</b><br>
            <small>score rerank: {r["score_rerank"]:.2f} | sim imagen: {r["sim"]:.3f}
            | id: {r["product_id"]}{marca}</small>
          </div>
        </div>"""

    opciones = [f"{r['product_id']} | {r['title'][:50]}" for r in resultados]
    return chat_visible, "", html, gr.Dropdown(choices=opciones, value=None,
                                               label="Selecciona un producto para dar feedback")

def dar_feedback(seleccion, tipo):
    if not seleccion:
        return "Selecciona un producto primero."
    pid = seleccion.split(" | ")[0]
    if tipo == "like":
        feedback["likes"].add(pid); feedback["dislikes"].discard(pid)
        return f"Me gusta registrado: {pid}. Afectara a las proximas busquedas."
    else:
        feedback["dislikes"].add(pid); feedback["likes"].discard(pid)
        return f"No me gusta registrado: {pid}. Afectara a las proximas busquedas."

def limpiar_feedback():
    feedback["likes"].clear(); feedback["dislikes"].clear()
    return "Feedback reiniciado."

def nueva_conversacion():
    global historial_chat
    historial_chat = []
    return [], "Conversacion reiniciada (memoria borrada)."

# ---------- Interfaz ----------
with gr.Blocks(title="RI Multimodal RAG") as demo:
    gr.Markdown("# Sistema de Recuperacion de Informacion Multimodal con RAG")
    with gr.Row():
        with gr.Column(scale=1):
            chatbox = gr.Chatbot(label="Conversacion", height=320)
            entrada = gr.Textbox(label="Tu mensaje",
                                 placeholder="ej: bright solar lights for my garden")
            with gr.Row():
                boton = gr.Button("Enviar", variant="primary")
                b_nueva = gr.Button("Nueva conversacion")
            gr.Markdown("### Relevance Feedback (Rocchio)")
            drop_fb = gr.Dropdown(choices=[], label="Selecciona un producto para dar feedback")
            with gr.Row():
                b_like = gr.Button("Me gusta")
                b_dislike = gr.Button("No me gusta")
                b_reset = gr.Button("Reiniciar feedback")
            estado_fb = gr.Textbox(label="Estado", lines=1)
        with gr.Column(scale=1):
            salida_ev = gr.HTML(label="Evidencias")

    boton.click(responder, inputs=[entrada, chatbox],
                outputs=[chatbox, entrada, salida_ev, drop_fb])
    entrada.submit(responder, inputs=[entrada, chatbox],
                   outputs=[chatbox, entrada, salida_ev, drop_fb])
    b_nueva.click(nueva_conversacion, outputs=[chatbox, estado_fb])
    b_like.click(lambda s: dar_feedback(s, "like"), inputs=drop_fb, outputs=estado_fb)
    b_dislike.click(lambda s: dar_feedback(s, "dislike"), inputs=drop_fb, outputs=estado_fb)
    b_reset.click(limpiar_feedback, outputs=estado_fb)

demo.launch(share=True, debug=False)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://2fa4165f528ef88318.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
